In [ ]:
from pathlib import Path
import pandas as pd, geopandas as gpd, duckdb

DATA = Path(r"C:\Users\erteo\Desktop\26T2\SIT378 - Project B")
SQL_BASE = "C:/Users/erteo/Desktop/26T2/SIT378 - Project B"   # forward slashes for SQL

con = duckdb.connect()

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)

In [ ]:
st = gpd.read_file(
    DATA / "260724 street spatial files" / "260724 street spatial files" / "streets_spatial.gpkg",
    layer="street_list_spatial_updated",
)
cbd = st[st.cbd == 1]
print(len(st), "segments,", len(cbd), "in CBD")
cbd[["street_name", "street_segment_id", "treatment_or_control", "intervention_type"]]

In [ ]:
con.sql(f"""
    SELECT StreetName, count(*) AS events
    FROM read_csv_auto('{SQL_BASE}/On-street_Car_Parking_Sensor_Data_-_2017.csv')
    GROUP BY 1 ORDER BY events DESC LIMIT 20
""").df()

In [ ]:
con.sql(f"""
    WITH y AS (
        SELECT 2016 AS yr, BetweenStreet1 b1, BetweenStreet2 b2,
               count(DISTINCT StreetMarker) bays
        FROM read_csv_auto('{SQL_BASE}/On-street_Car_Parking_Sensor_Data_-_2016.csv')
        WHERE StreetName = 'WILLIAM STREET' GROUP BY 1,2,3
        UNION ALL
        SELECT 2018, BetweenStreet1, BetweenStreet2, count(DISTINCT StreetMarker)
        FROM read_csv_auto('{SQL_BASE}/On-street_Car_Parking_Sensor_Data_-_2018.csv')
        WHERE StreetName = 'WILLIAM STREET' GROUP BY 1,2,3
    )
    SELECT b1, b2,
           max(CASE WHEN yr=2016 THEN bays END) AS bays_2016,
           max(CASE WHEN yr=2018 THEN bays END) AS bays_2018
    FROM y GROUP BY 1,2 ORDER BY 1,2
""").df()

## Why William Street, and why 2016 vs 2018

### Why the sensor analysis is limited to a few streets

The study covers 96 treatment segments, but the City of Melbourne sensor archive
only covers the CBD and only runs **Jan 2011 – May 2020**. Sensors were
decommissioned after that, so no data exists for any later period at any price.

Cross-referencing intervention dates against that window leaves **11 of 96**
treatment segments with usable data on both sides:

| Street | Segments | Intervention | Sensor coverage |
|---|---|---|---|
| William St | 4 | Apr 2017 | ~6 yrs pre, ~3 yrs post |
| La Trobe St | 7 | Jan 2013 | ~2 yrs pre, ~7 yrs post |
| Peel St | 2 | Jul 2020 | baseline only — intervention postdates the archive |
| Exhibition St | 2 | Oct 2020 | baseline only — intervention postdates the archive |

The remaining 85 treatment segments are outside the CBD or outside the sensor
window, and must be assessed from aerial imagery instead.

### Why William Street specifically

William Street is the strongest available case:

1. **Clear of COVID.** The April 2017 intervention has a full post-period ending
   well before March 2020, so no pandemic adjustment is needed.
2. **Long pre-period.** Sensors were mature by 2016, unlike La Trobe Street whose
   2011–2012 pre-period overlaps the sensor rollout and has unstable coverage.
3. **It is the intervention IV asked about** — a protected on-road bike lane, not
   a pedestrianisation or a regional reallocation.

La Trobe Street is analysed as a secondary case; Peel and Exhibition contribute
pre-intervention baselines whose post-period must come from Nearmap.

### Why 2016 and 2018, skipping 2017

The intervention is April 2017, so 2017 sits on both sides of it.

This matters because the bay count below uses `COUNT(DISTINCT StreetMarker)`
across a whole file. A bay removed in April 2017 would still appear in the 2017
data from its January–March activity, and would be counted as present — hiding
precisely the change being measured.

So only years falling wholly on one side are used:

- **2016** — entirely pre-intervention
- **2018** — entirely post-intervention, and past any construction period

Adjacent clean years are preferred over a wider gap (e.g. 2015 vs 2019) to limit
unrelated change: redevelopment, demand trends and restriction changes all
accumulate with time and would confound the comparison.

### Scope of this check

This is a **supply** check, not a utilisation result: it asks whether sensored
parking bays disappeared, not how heavily they were used. Utilisation is computed
separately using 12-month windows anchored on the actual intervention date, with a
30-day exclusion after it to avoid counting construction disruption.

### Limitations

- Sensors cover only part of each block (4–17 bays per block), so this measures
  the sensored subset of kerb, not total parking supply.
- A bay disappearing from the data could mean it was removed, or that its sensor
  was decommissioned. These cannot be distinguished from this dataset alone and
  must be checked against Nearmap imagery.
- Calendar years are an approximation used for this initial check only.

In [ ]:
def cols(year):
    return con.sql(f"""
        DESCRIBE SELECT * FROM read_csv_auto(
            '{SQL_BASE}/On-street_Car_Parking_Sensor_Data_-_{year}.csv')
    """).df().column_name.tolist()

for y in (2016, 2017, 2018):
    print(y, cols(y))

In [ ]:
def block_hours(year, street):
    """Hourly occupied and available minutes per block, from raw sensor events."""
    c = cols(year)
    present = '"Vehicle Present"' if "Vehicle Present" in c else "VehiclePresent"

    return con.sql(f"""
        WITH e AS (
            SELECT StreetMarker,
                   upper(replace(BetweenStreet1, 'Lt ', 'LITTLE ')) AS b1,
                   upper(replace(BetweenStreet2, 'Lt ', 'LITTLE ')) AS b2,
                   ArrivalTime AS a,
                   DepartureTime AS d,
                   {present} AS present
            FROM read_csv_auto('{SQL_BASE}/On-street_Car_Parking_Sensor_Data_-_{year}.csv')
            WHERE upper(replace(StreetName, 'Lt ', 'LITTLE ')) = '{street}'
              AND DepartureTime > ArrivalTime
              AND date_diff('hour', ArrivalTime, DepartureTime) < 24
        ),
        h AS (
            SELECT b1, b2, StreetMarker, a, d, present,
                   unnest(generate_series(date_trunc('hour', a), d, INTERVAL 1 HOUR)) AS hr
            FROM e
        ),
        m AS (
            SELECT b1, b2, hr, StreetMarker, present,
                   epoch(least(d, hr + INTERVAL 1 HOUR) - greatest(a, hr)) / 60 AS mins
            FROM h
        )
        SELECT b1, b2, hr,
               sum(CASE WHEN present THEN mins ELSE 0 END) AS occ_min,
               sum(mins)                                   AS avail_min,
               count(DISTINCT StreetMarker)                AS bays
        FROM m
        GROUP BY 1, 2, 3
        HAVING avail_min > 0
    """).df()


wm = pd.concat([block_hours(y, "WILLIAM STREET") for y in (2016, 2017, 2018)],
               ignore_index=True)
print(wm.shape, wm.columns.tolist())

In [ ]:
wm["hr"] = pd.to_datetime(wm.hr)
wm["util"] = wm.occ_min / wm.avail_min
core = wm[(wm.hr.dt.dayofweek < 5) & wm.hr.dt.hour.between(8, 18)]

cut = pd.Timestamp("2017-04-01")
pre  = core[core.hr.between(cut - pd.DateOffset(years=1), cut - pd.Timedelta(days=1))]
post = core[core.hr.between(cut + pd.Timedelta(days=30), cut + pd.DateOffset(years=1))]

res = pd.DataFrame({
    "util_pre":  pre.groupby(["b1","b2"]).util.mean(),
    "util_post": post.groupby(["b1","b2"]).util.mean(),
    "bays_pre":  pre.groupby(["b1","b2"]).bays.median(),
    "bays_post": post.groupby(["b1","b2"]).bays.median(),
}).round(3)
res["util_change_pp"] = ((res.util_post - res.util_pre) * 100).round(1)
res["bays_change"] = res.bays_post - res.bays_pre
res

## Method: from parking events to utilisation

### The problem

Each row in the sensor archive is one parking **event** — a vehicle arrived, then
departed. It is not an observation of a bay at a point in time. Utilisation can
therefore never be a row count: a busy hour and a quiet hour can contain the same
number of events.

A single event also spans multiple hours. A vehicle arriving at 09:40 and leaving
at 12:15 contributes to four different hours, in different amounts.

### Step 1 — Spread each event across the hours it touches

For every event, the number of occupied minutes falling inside each clock hour is
calculated by clipping the event to that hour's boundaries:

    occupied_minutes(hour) = min(departure, hour_end) − max(arrival, hour_start)

For the 09:40 → 12:15 example:

| Hour | Occupied minutes |
|---|---|
| 09:00 | 20 |
| 10:00 | 60 |
| 11:00 | 60 |
| 12:00 | 15 |

Total 155 minutes = 2h 35m, matching the original event duration.

Results are then summed to **block × hour**, along with a count of how many
distinct sensored bays were active in that hour.

### Step 2 — Convert to a utilisation rate

    utilisation(block, hour) = occupied_minutes / (bays × 60)

A block with 10 sensored bays offers 600 bay-minutes per hour. If 480 are
occupied, utilisation is 0.80. The measure is bounded 0–1; any value above 1
indicates a bug in the hour-splitting.

### Step 3 — Compare before and after

- **Weekdays only, 08:00–18:00.** Overnight hours are near-empty and would drag
  every average toward zero, masking differences in the periods that matter for
  commuting, retail and deliveries.
- **12-month windows either side** of the intervention date, so seasonal effects
  (summer holidays, school terms, weather) cancel out rather than confound.
- **30-day exclusion immediately after** the intervention date, so construction
  disruption is not counted as the post-intervention state.

### Data cleaning applied

Both are documented by the City of Melbourne as known issues:

- Events where departure is not after arrival are dropped (sensor logged the
  arrival after the departure).
- Events longer than 24 hours are dropped. Where a timestamp was not recorded,
  CoM back-fill it to midnight, which fabricates implausibly long stays.

### Reading the result — the critical point

Utilisation must always be read **alongside the bay count**, never alone:

| bays change | utilisation change | Interpretation |
|---|---|---|
| ↓ | ↑ | Supply shrank; the same demand now competes for fewer spaces. **Not** more parking. |
| — | ↓ | Supply unchanged; genuinely less parking demand. |
| — | — | No measurable parking effect from the intervention. |

Utilisation is a ratio, and the intervention can change its denominator. Reporting
a rise in utilisation without noting that bays were removed would invert the
policy conclusion — implying parking demand grew when the actual number of parked
vehicles may have fallen.

### Limitations

- Sensors cover only a subset of each block's kerb, so this measures sensored
  bays, not total parking supply.
- A bay vanishing from the data may mean it was physically removed, or that its
  sensor was decommissioned. The dataset cannot distinguish these; Nearmap imagery
  is required to confirm.
- Utilisation says nothing about who is parking or why. Turnover and stay duration
  are available from the same data and are worth reporting alongside.